# 90. 규정 승인 후 선택적 train+validation 최종 재훈련

**일반 개발 노트북이 아니다.** 운영 규정 또는 운영진 답변이 validation label의 최종
학습 사용을 명시적으로 허용한 경우에만 실행한다. 불명확하면 이 노트북을 닫고
2,000건 train artifact를 유지한다.

모든 출력은 `artifacts/final_train_validation/` 아래 새 namespace로 만들며 기존 OOF,
calibrator, stacker와 절대 혼합하지 않는다.


## 1. 독립 Colab 부트스트랩


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 2. 이중 승인 gate


In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

RULES_ALLOW_VALIDATION_LABEL_TRAINING = False
ACKNOWLEDGEMENT = ""
REQUIRED_ACKNOWLEDGEMENT = "규정이 validation label의 최종 학습 사용을 명시적으로 허용함을 확인했습니다"

RUN_COMBINED_DATASET = False
RUN_FINAL_FOLDS = False
RUN_FINAL_POLICY = False
PLAN_FINAL_TRAINING = False
EXECUTE_FINAL_TRAINING = False

FINAL_EXPERIMENT_ID = f"qwen3_scorer_final_2400_{RUN_NAMESPACE}"
FINAL_ROOT = PROJECT_ROOT / "artifacts" / "final_train_validation"
FINAL_FOLDS = FINAL_ROOT / "folds" / "folds_5fold_seed42.jsonl"
FINAL_POLICY = FINAL_ROOT / "policies" / "scorer_epoch3.json"

ANY_FINAL_ACTION = any([
    RUN_COMBINED_DATASET, RUN_FINAL_FOLDS, RUN_FINAL_POLICY,
    PLAN_FINAL_TRAINING, EXECUTE_FINAL_TRAINING,
])
if ANY_FINAL_ACTION:
    if not RULES_ALLOW_VALIDATION_LABEL_TRAINING:
        raise PermissionError("규정 승인 boolean이 False입니다.")
    if ACKNOWLEDGEMENT != REQUIRED_ACKNOWLEDGEMENT:
        raise PermissionError("승인 문구가 정확히 일치하지 않습니다.")
require_model_revision(MODEL_REVISION, enabled=PLAN_FINAL_TRAINING or EXECUTE_FINAL_TRAINING)
require_bf16_gpu(enabled=EXECUTE_FINAL_TRAINING)


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


## 3. immutable 2,400건 결합 artifact


In [ ]:
run_cli(
    "build final combined dataset",
    RUN_COMBINED_DATASET,
    "scripts/build_final_combined_data.py",
    "--config", "configs/data_final_combined.yaml",
    "--acknowledge-rules-allow-validation-label-training",
)


## 4. 전용 fold와 고정 epoch 정책


In [ ]:
run_cli(
    "final-only folds",
    RUN_FINAL_FOLDS,
    "scripts/make_folds.py",
    "--config", "configs/data_final_combined.yaml",
    "--n-folds", "5",
    "--seed", "42",
    "--output", FINAL_FOLDS,
)
run_cli(
    "final-only epoch policy",
    RUN_FINAL_POLICY,
    "scripts/select_fixed_epoch.py",
    "--preselected-epoch", "3",
    "--reason", "2,000건 개발 단계에서 잠근 최종 epoch 정책",
    "--output", FINAL_POLICY,
)


## 5. 전용 fold×seed plan 및 실행


In [ ]:
def final_training_arguments(*, execute: bool) -> list[object]:
    arguments: list[object] = [
        "scripts/orchestrate_scorer_training.py",
        "--config", "configs/scorer_qlora.yaml",
        "--data-config", "configs/data_final_combined.yaml",
        "--experiment-id", FINAL_EXPERIMENT_ID,
        "--folds-file", FINAL_FOLDS,
        "--epoch-policy", FINAL_POLICY,
        "--model-revision", MODEL_REVISION,
        "--seed", "42", "--seed", "1337", "--seed", "2026",
    ]
    if execute:
        arguments.append("--execute")
    arguments.extend(download_flag)
    return arguments

run_cli("final training plan", PLAN_FINAL_TRAINING, *final_training_arguments(execute=False))
run_cli("execute final training", EXECUTE_FINAL_TRAINING, *final_training_arguments(execute=True))


## 6. 이후 단계

새 fold×seed checkpoint로 OOF, calibrator, auxiliary source, stacker를 처음부터 다시 만든다.
2,000건 개발 artifact의 manifest나 scorer signature를 재사용하지 않는다. 이 경로의 성능
승격과 제출은 별도의 검증 결과가 있을 때만 허용한다.
